# YĀTRĀ AI — Notebook 05: Supervised Model Training
## Baseline Machine Learning Across 4 Estimator Families

This notebook implements supervised binary choice prediction:
$$\text{Target: } y_{ni} \in \{0, 1\} \quad (1 = \text{chosen alternative})$$
We train and evaluate four diverse model families:
1. **Regularized Logistic Regression** (Linear baseline, L2 penalty)
2. **Decision Tree Classifier** (Single interpretable tree, max_depth=6)
3. **Random Forest Classifier** (Bagged ensemble, 100 trees, max_depth=10)
4. **Histogram-based Gradient Boosting** (HistGradientBoosting, 100 iterations, learning_rate=0.1)

Evaluated across:
- **Core (Model A)**: 28 continuous and categorical features
- **Extended (Model B)**: 29 features (+ categorical `persona_type`)


In [ ]:
import os
import sys
import pandas as pd
import numpy as np

BASE_DIR = os.path.dirname(os.getcwd())
if BASE_DIR not in sys.path:
    sys.path.append(BASE_DIR)

from src.feature_engineering import split_by_traveller, build_feature_matrices
from src.train_models import train_all_models, get_model_specs
from src.evaluation import evaluate_partition

print("Training modules imported.")


### 1. Data Loading and Preprocessing


In [ ]:
df = pd.read_parquet(os.path.join(BASE_DIR, "data", "synthetic", "choice_dataset.parquet"))
df_train, df_val, df_test, _ = split_by_traveller(df, seed=42)
matrices_core = build_feature_matrices(df_train, df_val, df_test, feature_config="core")
print(f"Train samples: {len(df_train):,} | Validation samples: {len(df_val):,} | Test samples: {len(df_test):,}")


### 2. Inspecting Model Specifications and Hyperparameters


In [ ]:
specs = get_model_specs(seed=42)
for k, v in specs.items():
    print(f"{v['name']} ({k}):\n  Estimator: {v['estimator']}\n  Matrix Type: {v['matrix_type']}\n")


### 3. Baseline Model Benchmark Results (Pre-computed Authoritative Run)


In [ ]:
df_metrics = pd.read_csv(os.path.join(BASE_DIR, "results", "tables", "models", "model_metrics_table.csv"))
df_metrics[['model_identifier', 'feature_config', 'model_family', 'test_accuracy', 'test_precision', 'test_recall', 'test_f1', 'test_roc_auc']]


### 4. Overfitting Diagnostic Comparison
Comparing Train ROC-AUC vs. Validation and Test ROC-AUC to verify that regularized gradient boosting generalizes robustly.


In [ ]:
df_metrics[['model_identifier', 'model_family', 'train_roc_auc', 'val_roc_auc', 'test_roc_auc']]
